# Build subnational death-rate table (roadmap Step 5)

Vectorized (pandas/numpy) aggregation of the most current, geographically complete **subnational**
crude death rate, from two open sources joined to two geometry layers:

- **IHME GBD 2023** — all-cause, all-age, both-sex death rate (per 100k), 2023 — for ADM1 units
  worldwide (US states, Indian/Brazilian/Japanese/… regions). Joined to **Natural Earth 10m
  Admin-1** by `adm1_code`.
- **Eurostat `demo_r_gind3` (GDEATHRT)** — crude death rate by **NUTS-2**, 2023 — for the EU/EFTA +
  candidate countries GBD reports only nationally (Germany, France, Spain, the Nordics, Balkans…).
  Joined to **Eurostat GISCO NUTS-2** geometry by `NUTS_ID`.

**Output:** `data/subnational-cdr.json` — one flat `regions` list tagged by geometry layer
(`geo: "adm1" | "nuts2"`), plus `meta.nutsCountriesIso3` so the map can hide the Natural Earth
features that the finer NUTS layer draws over. **Roadmap-only** — does not feed the globe's
`rate-grid.json`. Rows are crude (not age-standardized): most of the between-region gap reflects
age structure, then real health differences.

In [2]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import requests


def find_root(start=None):
    """Walk upward until a directory containing package.json (mirrors notebooks/lib/grid.py)."""
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "package.json").exists():
            return candidate
    raise FileNotFoundError(f"No package.json found above {p}")


def norm(s):
    """Ascii-fold + lowercase + collapse non-alphanumerics, for name joins."""
    s = unicodedata.normalize("NFKD", str(s) if s is not None else "").encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()


ROOT = find_root()
SRC = ROOT / "data" / "source"
GBD_CSV = SRC / "IHME-GBD_2023_DATA-9789faec-1 (1)" / "IHME-GBD_2023_DATA-9789faec-1.csv"
NE_GEOJSON = SRC / "natural-earth" / "ne_10m_admin_1.geojson"
NUTS_GEOJSON = SRC / "natural-earth" / "nuts2_20m.geojson"
OUT_PATH = ROOT / "data" / "subnational-cdr.json"

YEAR = 2023
EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
    f"demo_r_gind3?format=JSON&time={YEAR}&indic_de=GDEATHRT"
)

for p in (GBD_CSV, NE_GEOJSON, NUTS_GEOJSON):
    assert p.exists(), f"missing input: {p}"
print("ROOT:", ROOT)
print("inputs present.")

ROOT: /Users/guillem/vault/projects/personal/watch-people-die-live
inputs present.


In [3]:
# GBD: all-cause, all-ages, both-sexes death RATE per 100k, latest year (vectorized filter).
# The CSV carries no parent-country column, only `location_name` — the geometry join infers it.
gbd = pd.read_csv(GBD_CSV)
gbd = gbd[
    (gbd["year"] == YEAR)
    & (gbd["metric_name"] == "Rate")
    & (gbd["measure_name"] == "Deaths")
].copy()
gbd = gbd.rename(columns={"location_name": "location", "val": "ratePer100k"})[["location", "ratePer100k"]]
gbd["norm"] = gbd["location"].map(norm)

# A few names repeat *within* GBD across different countries (e.g. "Distrito Federal" = Brazil
# AND Mexico) with different rates — name alone can't tell them apart, so drop those to avoid
# mis-coloring a region. Names that merely repeat with an identical rate collapse to one row.
rate_variety = gbd.groupby("norm")["ratePer100k"].transform("nunique")
ambiguous_in_gbd = sorted(gbd.loc[rate_variety > 1, "location"].unique())
gbd = gbd[rate_variety == 1].drop_duplicates("norm").reset_index(drop=True)

print(f"{len(gbd)} usable GBD {YEAR} locations. Dropped intra-GBD name collisions: {ambiguous_in_gbd}")
lookup = gbd.set_index("location")["ratePer100k"]
print({k: round(lookup[k]) for k in ["West Virginia", "Hawaii", "Utah"] if k in lookup})

789 usable GBD 2023 locations. Dropped intra-GBD name collisions: ['Georgia', 'Mexico', 'México', 'Niger', 'Punjab']
{'West Virginia': 1383, 'Hawaii': 913, 'Utah': 616}


In [4]:
# Natural Earth 10m Admin-1 -> a long table of (normalized name variant -> feature), so
# spelling/language variants still join. Each variant carries a `prio` (lower = more
# reliable): primary names beat aliases, so a feature whose `name_alt` happens to collide
# with another region's name (NE Tokyo's alt "Edo" == Nigeria's Edo) is never mis-assigned.
ne_props = pd.DataFrame([f["properties"] for f in json.loads(NE_GEOJSON.read_text())["features"]])
keep = ["adm1_code", "adm0_a3", "name", "iso_3166_2"]
name_fields = [("name", 0), ("name_en", 0), ("name_local", 1), ("gn_name", 2), ("woe_name", 3)]
ne_props = ne_props.reindex(columns=list(dict.fromkeys(keep + [c for c, _ in name_fields] + ["name_alt"])))

frames = [ne_props[keep].assign(variant=ne_props[col], prio=prio) for col, prio in name_fields]
# pipe-separated aliases are the least reliable (prio 4)
alt = ne_props[keep].assign(variant=ne_props["name_alt"].fillna("").str.split("|"), prio=4).explode("variant")
frames.append(alt)

ne_var = pd.concat(frames, ignore_index=True)
ne_var["norm"] = ne_var["variant"].map(norm)
ne_var = (
    ne_var[ne_var["norm"] != ""]
    .sort_values("prio")
    .drop_duplicates(["norm", "adm1_code"])  # best prio per (name, feature)
)

print(f"{ne_props['adm1_code'].nunique()} NE admin-1 features, {ne_var['norm'].nunique()} distinct name variants.")

4596 NE admin-1 features, 11906 distinct name variants.


In [5]:
# Match GBD names to Natural Earth features (vectorized merge), then disambiguate.
# Step 0: for each GBD name keep only its *best-priority* candidate features — a primary-name
#         hit (prio 0) always wins over an alias hit, so alias collisions can't mis-assign.
# Step 1: names hitting exactly one country resolve directly and reveal which countries GBD
#         reports subnationally (>= 3 confidently-matched regions).
# Step 2: remaining collisions go to the candidate country in that subnational set, with a
#         small override table for true ties.
OVERRIDES = {  # normalized gbd name -> adm0_a3
    norm("Distrito Federal"): "BRA",
    norm("South West"): "CMR",
    norm("Luxembourg"): "BEL",
}

cand = gbd.merge(ne_var[["norm", "adm1_code", "adm0_a3", "name", "iso_3166_2", "prio"]], on="norm", how="inner")
cand = cand[cand["prio"] == cand.groupby("location")["prio"].transform("min")]  # best-priority hits only

# Pass 1: unambiguous names + the resulting GBD-subnational country set.
cand["unambiguous"] = cand.groupby("location")["adm0_a3"].transform("nunique") == 1
unamb = cand[cand["unambiguous"]]
sub_counts = unamb.groupby("adm0_a3")["adm1_code"].nunique()
gbd_sub_countries = set(sub_counts[sub_counts >= 3].index)

# Pass 2: resolve the ambiguous rows.
cand["override_ok"] = cand["norm"].map(OVERRIDES).eq(cand["adm0_a3"])
amb = cand[~cand["unambiguous"] & ~cand["override_ok"]]
amb_in_sub = amb[amb["adm0_a3"].isin(gbd_sub_countries)]
resolvable = amb_in_sub[amb_in_sub.groupby("location")["adm0_a3"].transform("nunique") == 1]

chosen = pd.concat([unamb, cand[cand["override_ok"]], resolvable], ignore_index=True)
# One feature matched by >1 GBD name: keep the best priority, then deterministic by name.
adm1 = (
    chosen.sort_values(["adm1_code", "prio", "location"])
    .drop_duplicates("adm1_code")[["adm1_code", "name", "adm0_a3", "iso_3166_2", "ratePer100k"]]
    .reset_index(drop=True)
    .rename(columns={"adm1_code": "key", "adm0_a3": "country"})
)
adm1["geo"] = "adm1"

still_amb = sorted(set(amb["location"]) - set(resolvable["location"]) - set(cand[cand["override_ok"]]["location"]))
print(f"GBD-subnational countries: {len(gbd_sub_countries)} | matched regions: {len(adm1)} | still ambiguous: {still_amb}")
print("Tokyo check:", adm1.loc[adm1["key"] == "JPN-1860", "ratePer100k"].round(1).tolist())

GBD-subnational countries: 17 | matched regions: 550 | still ambiguous: []
Tokyo check: [988.7]


In [6]:
# Eurostat demo_r_gind3 (GDEATHRT = crude death rate per 1,000) at NUTS-2, joined to GISCO
# NUTS-2 geometry by NUTS_ID. Fills the EU/EFTA + candidate countries GBD reports nationally.
resp = requests.get(EUROSTAT_URL, timeout=90).json()
geo_pos = pd.Series(resp["dimension"]["geo"]["category"]["index"])  # NUTS code -> flat position
values = pd.Series(resp["value"])  # position (str) -> value; other dims are size-1 so pos == geo pos

euro = geo_pos.rename("pos").reset_index().rename(columns={"index": "NUTS_ID"})
euro = euro[euro["NUTS_ID"].str.len() == 4]  # len 4 == NUTS-2
euro["cdrPer1000"] = euro["pos"].astype(str).map(values)
euro = euro.dropna(subset=["cdrPer1000"]).drop(columns="pos")

nuts_props = pd.DataFrame(
    [f["properties"] for f in json.loads(NUTS_GEOJSON.read_text())["features"]]
)[["NUTS_ID", "NAME_LATN", "ISO3_CODE", "CNTR_CODE"]]

nuts = euro.merge(nuts_props, on="NUTS_ID", how="inner").dropna(subset=["ISO3_CODE"])
nuts = nuts.rename(columns={"NAME_LATN": "name", "ISO3_CODE": "country", "NUTS_ID": "key"})
nuts["ratePer100k"] = nuts["cdrPer1000"] * 100.0
nuts["geo"] = "nuts2"

nuts_countries_iso3 = sorted(nuts["country"].unique())
print(f"Eurostat NUTS-2 regions with a {YEAR} CDR + geometry: {len(nuts)} "
      f"across {len(nuts_countries_iso3)} countries.")
print("highest:", nuts.nlargest(3, "ratePer100k")[["name", "cdrPer1000"]].values.tolist())
print("lowest: ", nuts.nsmallest(3, "ratePer100k")[["name", "cdrPer1000"]].values.tolist())

Eurostat NUTS-2 regions with a 2023 CDR + geometry: 287 across 36 countries.
highest: [['Severozapaden', 20.4], ['Severen tsentralen', 18.6], ['Sachsen-Anhalt', 16.6]]
lowest:  [['Mardin, Batman, Şırnak, Siirt', 2.9], ['Mayotte', 3.0], ['Van, Muş, Bitlis, Hakkari', 3.1]]


In [7]:
# National fallback: World Bank crude death rate per country (data/source/cdr-snapshot.json,
# the same national figure the globe uses), keyed by ISO3 so the map can color countries that
# have no regional data — and fill regional gaps inside partly-covered countries — at their
# national rate. cdr value is per 1,000; x100 -> per 100k to match the regional scale.
cdr_raw = json.loads((ROOT / "data" / "source" / "cdr-snapshot.json").read_text())["values"]
country_rates = {
    row["iso3"]: round(row["value"] * 100.0, 1)
    for row in cdr_raw
    if row.get("iso3") and row.get("value") is not None
}
print(f"National CDR fallback: {len(country_rates)} countries (World Bank).")
print("e.g.", {k: country_rates[k] for k in ["CHN", "RUS", "DEU", "USA"] if k in country_rates})

National CDR fallback: 169 countries (World Bank).
e.g. {'CHN': 776.0, 'RUS': 1240.0, 'DEU': 1210.0, 'USA': 900.0}


## National statistical offices — five countries GBD/Eurostat miss subnationally

IHME GBD reports **Russia, Canada, Argentina, Australia, Chile** only nationally, and they
sit outside the Eurostat NUTS layer — so Step 5 flattened each to a single national color.
Here we add their real first-level crude death rates from each country's own statistical
office, joined to Natural Earth Admin-1 by **`iso_3166_2`** (language-neutral, stable):

- **Canada** — StatCan Table 13-10-0710, *Mortality rate per 1,000 population*, 2023 (direct).
- **Australia** — ABS `DEATHS_SUMMARY` (cat. 3302.0), MEASURE 5 = crude rate per 1,000, 2023.
- **Russia** — Rosstat *Естественное движение населения* `EDN_2023.xlsx`, tab `ТАБ_2` col F
  (умершие на 1000), 2023. Arkhangelsk/Tyumen use the *без автономии* rows so Nenets / KhMAO /
  YaNAO stay separate features; Crimea/Sevastopol are coded `UA` in the geometry so are not mapped.
- **Argentina** — DEIS deaths by residence (2022, most-complete year) ÷ INDEC provincial
  population projections 2022 → crude rate per 1,000.
- **Chile** — DEIS *Defunciones por Semana Epidemiológica* (2022): Σ deaths ÷ INE population
  (one week, since population repeats weekly) → crude rate per 1,000.

Also aliases Natural Earth's non-ISO country codes **`SDS`→`SSD`** (South Sudan) and
**`PSX`→`PSE`** (Palestine) onto the World Bank national fallback, which otherwise never
resolves and leaves those countries uncolored.

Raw sources live in `data/source/subnational/` (gitignored).

In [10]:
# Five countries GBD reports only nationally + Eurostat doesn't cover: pull each country's own
# statistical office and join to Natural Earth Admin-1 by iso_3166_2. Each block yields
# {iso_3166_2: cdrPer1000}; the shared join turns that into adm1 rows keyed by adm1_code.
SUB = SRC / "subnational"

# iso_3166_2 -> (adm1_code, adm0_a3, NE name) from the geometry already loaded in cell 3.
iso_geo = (
    ne_props.dropna(subset=["iso_3166_2"])
    .drop_duplicates("iso_3166_2")
    .set_index("iso_3166_2")[["adm1_code", "adm0_a3", "name"]]
)

# ---- Canada: StatCan 13-10-0710, "Mortality rate per 1,000 population", all ages, both sexes, 2023
CA_NAME2ISO = {
    "Newfoundland and Labrador": "CA-NL", "Prince Edward Island": "CA-PE", "Nova Scotia": "CA-NS",
    "New Brunswick": "CA-NB", "Quebec": "CA-QC", "Ontario": "CA-ON", "Manitoba": "CA-MB",
    "Saskatchewan": "CA-SK", "Alberta": "CA-AB", "British Columbia": "CA-BC", "Yukon": "CA-YT",
    "Northwest Territories": "CA-NT", "Nunavut": "CA-NU",
}
_ca = pd.read_csv(SUB / "13100710.csv", low_memory=False)
_ca = _ca[(_ca["Age at time of death"] == "Age at time of death, all ages")
          & (_ca["Sex"] == "Both sexes")
          & (_ca["Characteristics"] == "Mortality rate per 1,000 population")
          & (_ca["REF_DATE"] == 2023)]
canada = {CA_NAME2ISO[n]: round(float(v), 1)
          for n, v in zip(_ca["GEO"].str.replace(", place of residence", "", regex=False).str.strip(),
                          _ca["VALUE"]) if n in CA_NAME2ISO}

# ---- Australia: ABS DEATHS_SUMMARY, MEASURE 5 = crude death rate per 1,000, persons (SEX 3), 2023
AU_CODE2ISO = {1: "AU-NSW", 2: "AU-VIC", 3: "AU-QLD", 4: "AU-SA",
               5: "AU-WA", 6: "AU-TAS", 7: "AU-NT", 8: "AU-ACT"}
_au = pd.read_csv(SUB / "australia-deaths.csv")
_au = _au[(_au.MEASURE == 5) & (_au.SEX == 3) & (_au.TIME_PERIOD == 2023)]
_au = _au[pd.to_numeric(_au.REGION, errors="coerce").notna()]
australia = {AU_CODE2ISO[int(r)]: round(float(v), 1)
             for r, v in zip(_au.REGION, _au.OBS_VALUE) if int(r) in AU_CODE2ISO}

# ---- Russia: Rosstat EDN_2023.xlsx, tab ТАБ_2, col F = deaths per 1,000, 2023
_LAT2CYR = str.maketrans({"A": "А", "B": "В", "E": "Е", "K": "К", "M": "М", "H": "Н", "O": "О",
                          "P": "Р", "C": "С", "T": "Т", "X": "Х", "Y": "У", "a": "а", "e": "е",
                          "k": "к", "m": "м", "o": "о", "p": "р", "c": "с", "t": "т", "x": "х", "y": "у"})
def ru_norm(s):
    s = unicodedata.normalize("NFKC", str(s)).strip().lower().translate(_LAT2CYR)
    s = re.sub(r"^г\.?\s*", "", s)          # drop city prefix "г."
    s = re.sub(r"[—–]", "-", s)             # normalize dashes
    return re.sub(r"\s+", " ", s)
RU_NAME2ISO = {ru_norm(k): v for k, v in {
    "Белгородская область": "RU-BEL", "Брянская область": "RU-BRY", "Владимирская область": "RU-VLA",
    "Воронежская область": "RU-VOR", "Ивановская область": "RU-IVA", "Калужская область": "RU-KLU",
    "Костромская область": "RU-KOS", "Курская область": "RU-KRS", "Липецкая область": "RU-LIP",
    "Московская область": "RU-MOW", "Орловская область": "RU-ORL", "Рязанская область": "RU-RYA",
    "Смоленская область": "RU-SMO", "Тамбовская область": "RU-TAM", "Тверская область": "RU-TVE",
    "Тульская область": "RU-TUL", "Ярославская область": "RU-YAR", "Москва": "RU-MOS",
    "Республика Карелия": "RU-KR", "Республика Коми": "RU-KO",
    "Архангельская область без автономии": "RU-ARK", "Ненецкий автономный округ": "RU-NEN",
    "Вологодская область": "RU-VLG", "Калининградская область": "RU-KGD",
    "Ленинградская область": "RU-LEN", "Мурманская область": "RU-MUR",
    "Новгородская область": "RU-NGR", "Псковская область": "RU-PSK", "Санкт-Петербург": "RU-SPE",
    "Республика Адыгея": "RU-AD", "Республика Калмыкия": "RU-KL", "Краснодарский край": "RU-KDA",
    "Астраханская область": "RU-AST", "Волгоградская область": "RU-VGG", "Ростовская область": "RU-ROS",
    "Республика Дагестан": "RU-DA", "Республика Ингушетия": "RU-IN",
    "Кабардино-Балкарская Республика": "RU-KB", "Карачаево-Черкесская Республика": "RU-KC",
    "Республика Северная Осетия-Алания": "RU-SE", "Чеченская Республика": "RU-CE",
    "Ставропольский край": "RU-STA", "Республика Башкортостан": "RU-BA", "Республика Марий Эл": "RU-ME",
    "Республика Мордовия": "RU-MO", "Республика Татарстан": "RU-TA", "Удмуртская Республика": "RU-UD",
    "Чувашская Республика": "RU-CU", "Пермский край": "RU-PER", "Кировская область": "RU-KIR",
    "Нижегородская область": "RU-NIZ", "Оренбургская область": "RU-ORE", "Пензенская область": "RU-PNZ",
    "Самарская область": "RU-SAM", "Саратовская область": "RU-SAR", "Ульяновская область": "RU-ULY",
    "Курганская область": "RU-KGN", "Свердловская область": "RU-SVE",
    "Тюменская область без автономий": "RU-TYU", "Ханты-Мансийский автономный округ - Югра": "RU-KHM",
    "Ямало-Ненецкий автономный округ": "RU-YAN", "Челябинская область": "RU-CHE",
    "Республика Алтай": "RU-AL", "Республика Тыва": "RU-TY", "Республика Хакасия": "RU-KK",
    "Алтайский край": "RU-ALT", "Красноярский край": "RU-KYA", "Иркутская область": "RU-IRK",
    "Кемеровская область": "RU-KEM", "Новосибирская область": "RU-NVS", "Омская область": "RU-OMS",
    "Томская область": "RU-TOM", "Республика Бурятия": "RU-BU", "Республика Саха (Якутия)": "RU-SA",
    "Забайкальский край": "RU-ZAB", "Камчатский край": "RU-KAM", "Приморский край": "RU-PRI",
    "Хабаровский край": "RU-KHA", "Амурская область": "RU-AMU", "Магаданская область": "RU-MAG",
    "Сахалинская область": "RU-SAK", "Еврейская автономная область": "RU-YEV",
    "Чукотский автономный округ": "RU-CHU",
}.items()}
_ru = pd.read_excel(SUB / "russia-EDN_2023.xlsx", sheet_name="ТАБ_2", header=None)
russia = {}
for i in range(5, _ru.shape[0]):
    nm, val = _ru.iloc[i, 0], _ru.iloc[i, 5]
    key = ru_norm(nm) if pd.notna(nm) else None
    if key in RU_NAME2ISO and pd.notna(val):
        russia[RU_NAME2ISO[key]] = round(float(val), 1)

# ---- Argentina: DEIS deaths by residence (2022) / INDEC provincial population (2022)
AR_ID2ISO = {2: "AR-C", 6: "AR-B", 10: "AR-K", 14: "AR-X", 18: "AR-W", 22: "AR-H", 26: "AR-U",
             30: "AR-E", 34: "AR-P", 38: "AR-Y", 42: "AR-L", 46: "AR-F", 50: "AR-M", 54: "AR-N",
             58: "AR-Q", 62: "AR-R", 66: "AR-A", 70: "AR-J", 74: "AR-D", 78: "AR-Z", 82: "AR-S",
             86: "AR-G", 90: "AR-T", 94: "AR-V"}
_ar = pd.read_csv(SUB / "argentina-def-2005-2022.csv",
                  usecols=["anio", "jurisdiccion_de_residencia_id", "cantidad"], low_memory=False)
_ar = _ar[(_ar.anio == 2022) & (_ar.jurisdiccion_de_residencia_id.isin(AR_ID2ISO))]
_ar_deaths = _ar.groupby("jurisdiccion_de_residencia_id")["cantidad"].sum()
_arx = pd.ExcelFile(SUB / "argentina-indec-proj.xls")
_ar_sheet = {int(s[:2]): s for s in _arx.sheet_names if "-" in s and s[:2].isdigit()}
def _ar_pop2022(sheet):
    df = pd.read_excel(_arx, sheet_name=sheet, header=None)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            if str(df.iloc[i, j]).strip() in ("2022", "2022.0"):
                for k in range(i, min(i + 8, df.shape[0])):
                    if str(df.iloc[k, 0]).strip().lower() == "total":
                        return float(df.iloc[k, j])
    return None
argentina = {}
for code, iso in AR_ID2ISO.items():
    pop = _ar_pop2022(_ar_sheet[code]) if code in _ar_sheet else None
    d = int(_ar_deaths.get(code, 0))
    if pop and pop > 0:
        argentina[iso] = round(d / pop * 1000, 1)

# ---- Chile: DEIS Defunciones por Semana Epidemiológica (deaths + INE population, 2022)
CL_NAME2ISO = {
    "De Aisén del Gral. C. Ibáñez del Campo": "CL-AI", "De Antofagasta": "CL-AN",
    "De Arica y Parinacota": "CL-AP", "De Atacama": "CL-AT", "De Coquimbo": "CL-CO",
    "De La Araucanía": "CL-AR", "De Los Lagos": "CL-LL", "De Los Ríos": "CL-LR",
    "De Magallanes y de La Antártica Chilena": "CL-MA", "De Tarapacá": "CL-TA",
    "De Valparaíso": "CL-VS", "De Ñuble": "CL-NB", "Del Bíobío": "CL-BI",
    "Del Libertador B. O'Higgins": "CL-LI", "Del Maule": "CL-ML", "Metropolitana de Santiago": "CL-RM",
}
_cl = pd.read_csv(SUB / "chile-def-semana.csv", sep="|")
_cl = _cl[_cl.ANO_ESTADISTICO.astype(str).str.fullmatch(r"\d+")]
_cl["ANO_ESTADISTICO"] = _cl.ANO_ESTADISTICO.astype(int)
_cl = _cl[(_cl.ANO_ESTADISTICO == 2022) & (_cl.REGION.isin(CL_NAME2ISO))]
_cl_deaths = _cl.groupby("REGION")["MUERTES_OBS"].sum()
_cl_pop = _cl[_cl.SEMANA_ESTADISTICA == _cl.SEMANA_ESTADISTICA.min()].groupby("REGION")["POBLACION"].sum()
chile = {CL_NAME2ISO[r]: round(_cl_deaths[r] / _cl_pop[r] * 1000, 1)
         for r in CL_NAME2ISO if r in _cl_deaths.index and _cl_pop.get(r, 0) > 0}

# ---- join each {iso: cdr} to geometry -> adm1 rows (same schema as `adm1`) ----
NSO_YEARS = {"CAN": 2023, "AUS": 2023, "RUS": 2023, "ARG": 2022, "CHL": 2022}
_expected = {"canada": 13, "australia": 8, "russia": 83, "argentina": 24, "chile": 16}
_rows, _cov = [], {}
for tag, rates in [("canada", canada), ("australia", australia), ("russia", russia),
                   ("argentina", argentina), ("chile", chile)]:
    hit = 0
    for iso, cdr in rates.items():
        if iso in iso_geo.index:
            g = iso_geo.loc[iso]
            _rows.append({"geo": "adm1", "key": g["adm1_code"], "name": g["name"],
                          "country": g["adm0_a3"], "ratePer100k": round(cdr * 100.0, 1)})
            hit += 1
    _cov[tag] = (len(rates), hit)
    assert hit == _expected[tag], f"{tag}: joined {hit}, expected {_expected[tag]} (rates={len(rates)})"
nso_adm1 = pd.DataFrame(_rows)
assert nso_adm1["ratePer100k"].between(300, 2500).all(), "NSO rate out of sane band"
assert not nso_adm1["key"].duplicated().any(), "duplicate adm1 key across NSO sources"

# South Sudan / Palestine: Natural Earth codes them SDS/PSX, but the World Bank fallback is
# keyed by ISO3 SSD/PSE, so the map's national-rate lookup missed and left them uncolored.
for ne_code, iso3 in [("SDS", "SSD"), ("PSX", "PSE")]:
    if iso3 in country_rates:
        country_rates[ne_code] = country_rates[iso3]

print("NSO subnational coverage (rates -> joined to geometry):")
for tag, (n, hit) in _cov.items():
    print(f"  {tag:10s} {hit:3d}/{n}")
print(f"  total new adm1 rows: {len(nso_adm1)}")
print("country-code aliases added to fallback:",
      {c: country_rates.get(c) for c in ["SDS", "PSX"]})

NSO subnational coverage (rates -> joined to geometry):
  canada      13/13
  australia    8/8
  russia      83/83
  argentina   24/24
  chile       16/16
  total new adm1 rows: 144
country-code aliases added to fallback: {'SDS': 982.1, 'PSX': 587.0}


In [11]:
# Combine all geometry layers into one flat regions list and write data/subnational-cdr.json.
# adm1 = GBD-matched regions + the five national-statistical-office countries (nso_adm1);
# drop_duplicates(keep="last") lets an NSO row win over any GBD name-collision on the same feature.
cols = ["geo", "key", "name", "country", "cdrPer1000", "ratePer100k"]
adm1_all = pd.concat([adm1, nso_adm1], ignore_index=True).drop_duplicates("key", keep="last")
adm1_out = adm1_all.assign(cdrPer1000=adm1_all["ratePer100k"] / 100.0)[cols]
regions_df = pd.concat([adm1_out, nuts[cols]], ignore_index=True)
regions_df["cdrPer1000"] = regions_df["cdrPer1000"].round(4)
regions_df["ratePer100k"] = regions_df["ratePer100k"].round(1)

out = {
    "meta": {
        "sources": [
            "IHME Global Burden of Disease 2023 (all-cause, all-age, both-sex crude death rate) -> Natural Earth 10m Admin-1",
            f"Eurostat demo_r_gind3 GDEATHRT crude death rate, NUTS-2, {YEAR} -> GISCO NUTS-2 geometry",
            "National statistical offices, first-level regions -> Natural Earth 10m Admin-1 by iso_3166_2: "
            "StatCan Table 13-10-0710 (Canada 2023); ABS DEATHS_SUMMARY cat. 3302.0 (Australia 2023); "
            "Rosstat EDN_2023 Естественное движение населения (Russia 2023); DEIS + INDEC (Argentina 2022); "
            "DEIS + INE (Chile 2022)",
            "World Bank crude death rate per country (national fallback where no regional data)",
        ],
        "year": YEAR,
        "unit": "cdrPer1000 = deaths per 1,000 population/yr; ratePer100k = per 100,000",
        "note": "Crude (not age-standardized): most of the between-region gap reflects age structure, "
        "then real health differences. Roadmap-only; does not feed the globe rate-grid.",
        "geoLayers": {"adm1": "data/admin1-10m.json", "nuts2": "data/nuts2-20m.json"},
        "nutsCountriesIso3": nuts_countries_iso3,  # hide these Natural Earth features (drawn as NUTS)
        "subnationalOfficeYears": NSO_YEARS,  # per-country reference year for the NSO-sourced regions
        "regionCount": int(len(regions_df)),
        "adm1Count": int((regions_df["geo"] == "adm1").sum()),
        "nuts2Count": int((regions_df["geo"] == "nuts2").sum()),
        "countryCount": int(regions_df["country"].nunique()),
        "countryFallbackCount": len(country_rates),
        "license": "GBD free-use agreement (non-commercial); Eurostat open reuse; "
        "national statistical offices (StatCan/ABS/Rosstat/DEIS/INDEC/INE) open reuse; World Bank.",
    },
    "regions": regions_df.to_dict("records"),
    "countryRates": country_rates,  # ISO3 (+ NE SDS/PSX aliases) -> ratePer100k national fallback
}
OUT_PATH.write_text(json.dumps(out, default=float))
print(f"Wrote {OUT_PATH.relative_to(ROOT)}: {out['meta']['regionCount']} regions "
      f"({out['meta']['adm1Count']} adm1 + {out['meta']['nuts2Count']} nuts2) across "
      f"{out['meta']['countryCount']} countries ({OUT_PATH.stat().st_size / 1024:.0f} KB).")

Wrote data/subnational-cdr.json: 981 regions (694 adm1 + 287 nuts2) across 72 countries (119 KB).


In [12]:
# Verification + showcase: biggest within-country spread (max/min region) per country with
# >= 5 mapped regions, across both layers. These drive the roadmap chart's callouts.
allr = regions_df.copy()
grp = allr.groupby("country")
spread = pd.DataFrame({
    "n": grp["ratePer100k"].size(),
    "hi": grp["ratePer100k"].max(),
    "lo": grp["ratePer100k"].min(),
    "hi_name": grp.apply(lambda d: d.loc[d["ratePer100k"].idxmax(), "name"], include_groups=False),
    "lo_name": grp.apply(lambda d: d.loc[d["ratePer100k"].idxmin(), "name"], include_groups=False),
})
spread = spread[spread["n"] >= 5]
spread["ratio"] = spread["hi"] / spread["lo"]
with pd.option_context("display.width", 140, "display.max_rows", 20):
    print(spread.sort_values("ratio", ascending=False).head(16).round(2).to_string())

# Assertions
assert len(regions_df) > 800, "expected 550 adm1 + ~290 nuts2"
assert (regions_df["ratePer100k"] > 0).all(), "non-positive rate"
assert not regions_df.duplicated("key").any(), "duplicate region key"
assert {"DEU", "FRA", "ESP", "SWE"} <= set(regions_df["country"]), "EU coverage missing"
print("\nOK: assertions passed.")

          n      hi     lo                         hi_name                        lo_name  ratio
country                                                                                         
RUS      83  1700.0  320.0                           Pskov                         Ingush   5.31
TUR      26  1450.0  290.0  Hatay, Kahramanmaraş, Osmaniye  Mardin, Batman, Şırnak, Siirt   5.00
FRA      27  1300.0  300.0                        Limousin                        Mayotte   4.33
PHL      79   841.8  210.6                         Batanes                  Lanao del Sur   4.00
IND      28  1198.8  374.0                       Telangana              Arunachal Pradesh   3.21
ARG      24  1080.0  420.0          Ciudad de Buenos Aires               Tierra del Fuego   2.57
USA      50  1383.1  615.5                   West Virginia                           Utah   2.25
ESP      19  1290.0  580.0          Principado de Asturias              Ciudad de Melilla   2.22
IRN      29   654.9  319.8    